# 05 - Evaluation

This notebook compares model outputs and translates metrics into project findings. It uses the CSV files saved by Notebook 04.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"

## Load Model Results

In [ ]:
clustering_metrics = pd.read_csv(OUTPUT_DIR / "clustering_metrics.csv")
cluster_profile = pd.read_csv(OUTPUT_DIR / "cluster_profile.csv")
regression_metrics = pd.read_csv(OUTPUT_DIR / "regression_metrics.csv")
classification_metrics = pd.read_csv(OUTPUT_DIR / "classification_metrics.csv")
anomaly_results = pd.read_csv(OUTPUT_DIR / "anomaly_results.csv", parse_dates=["datetime"])
pca_summary = pd.read_csv(OUTPUT_DIR / "pca_summary.csv")

display(clustering_metrics)
display(cluster_profile)
display(regression_metrics)
display(classification_metrics)

## Clustering Evaluation

Silhouette scores measure how separated clusters are. Higher is better, but household energy behavior can be naturally overlapping because many appliance patterns happen at similar times.

In [ ]:
best_clustering = clustering_metrics.sort_values("silhouette", ascending=False).iloc[0]
print("Best clustering row:")
display(best_clustering.to_frame().T)

plt.figure(figsize=(8, 4))
sns.barplot(data=clustering_metrics, x="algorithm", y="silhouette", hue="clusters")
plt.title("Clustering Silhouette Comparison")
plt.tight_layout()
plt.show()

## Regression Evaluation

Regression is evaluated using MAE, RMSE, and R2. RMSE is useful here because it penalizes large prediction errors, which matter for energy spike prediction.

In [ ]:
best_regression = regression_metrics.sort_values("rmse").iloc[0]
print("Best regression model:", best_regression["model_name"])
display(best_regression.to_frame().T)

plt.figure(figsize=(8, 4))
sns.barplot(data=regression_metrics, x="model_name", y="rmse")
plt.title("Regression RMSE Comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Classification Evaluation

Classification is evaluated with accuracy and the full classification report saved by the modeling notebook.

In [ ]:
best_classification = classification_metrics.sort_values("accuracy", ascending=False).iloc[0]
print("Best classification model:", best_classification["model_name"])
display(best_classification.to_frame().T)

report_text = (OUTPUT_DIR / "classification_report.txt").read_text(encoding="utf-8")
print(report_text)

## Anomaly and PCA Evaluation

In [ ]:
anomaly_share = anomaly_results["anomaly_label"].mean() * 100
print(f"Detected anomaly share: {anomaly_share:.2f}%")

top_anomalies = anomaly_results.sort_values("anomaly_score").head(10)
display(top_anomalies[["datetime", "global_active_power", "global_intensity", "anomaly_score"]])
display(pca_summary)

## Final Evaluation Summary

- Clustering gives useful behavior segments, although household energy patterns overlap.
- Regression provides moderate short-term prediction performance using time and lag features.
- Classification performs strongly for high-vs-normal consumption periods.
- Anomaly detection highlights rare high-power events worth reviewing.
- The dashboard is appropriate as the deployment layer because it turns these results into interactive filters and visuals.